In [39]:
import numpy as np
import pandas as pd

In [40]:
data_ids = [361260, 361259, 361242]
n_ests = [50, 100, 500, 1000]
min_samples_leafs = [1, 5, 10]
max_features = [0.1, 0.33, "1.0"]

In [41]:
# for each data_id, load the result and save as a df
dfs = []
for data_id in data_ids:
    # get number of samples in the data_id by reading X csv
    X = np.loadtxt(f"data/{data_id}/X.csv", delimiter=",")
    n_samples = 1000
    n_features = X.shape[1]
    for n_est in n_ests:
        for min_samples_leaf in min_samples_leafs:
            for max_feature in max_features:
                # create the directory if it doesn't exist
                dir_path = f"results/{data_id}/n_estimators_{n_est}/min_samples_leaf_{min_samples_leaf}/max_features_{max_feature}"
                results_path = f"{dir_path}/runtime_results.csv"
                results_df = pd.read_csv(results_path)
                # divide every col in df except 'data_id' by n_samples
                for col in results_df.columns:
                    if col != 'data_id':
                        results_df[col] = results_df[col] / n_samples
                # add columns for n_estimators, min_samples_leaf, max_features
                results_df['n_estimators'] = n_est
                results_df['min_samples_leaf'] = min_samples_leaf
                results_df['max_features'] = max_feature
                results_df['num_features'] = n_features
                dfs.append(results_df)
df = pd.concat(dfs, ignore_index=True)

In [42]:
df[(df['n_estimators'] == 100) & (df['min_samples_leaf'] == 5) & (df['max_features'] == 0.33)]

,data_id,rf_fitting_time,rf_plus_baseline_fitting_time,rf_plus_fitting_time,shap_explainer_time,shap_values_time,lime_time,lmdi_baseline_explainer_time,lmdi_baseline_values_time,lmdi_plus_explainer_time,lmdi_plus_values_time,n_estimators,min_samples_leaf,max_features,num_features
13,361260,0.000240,0.008094,0.007386,0.000005,0.002666,0.082713,0.000006,0.002627,0.000014,0.002818,100,5,0.33,15
49,361259,0.000641,0.005609,0.010538,0.000012,0.003305,0.129364,0.000007,0.008894,0.000022,0.010962,100,5,0.33,32
85,361242,0.001334,0.009553,0.017130,0.000011,0.003984,0.273451,0.000007,0.027807,0.000010,0.035832,100,5,0.33,81


In [43]:
display_df = df[(df['n_estimators'] == 100) & (df['min_samples_leaf'] == 5) & (df['max_features'] == 0.33)]
# display_df columns should be data_id, n_features, n_estimators, min_samples_leaf, max_features, lime_time, shap_values_time, rf_plus_fitting_time + lmdi_plus_values_time
display_df = display_df[['data_id', 'num_features', 'lime_time', 'shap_values_time', 'rf_plus_fitting_time', 'lmdi_plus_values_time']]
display_df['lmdi_plus_time'] = display_df['rf_plus_fitting_time'] + display_df['lmdi_plus_values_time']
display_df.drop(columns=['rf_plus_fitting_time', 'lmdi_plus_values_time'], inplace=True)
display_df = display_df.rename(columns={
    'data_id': 'OpenML Data ID',
    'num_features': '# of Features',
    'n_estimators': '# of Estimators',
    'min_samples_leaf': 'Min. Samples per Leaf',
    'max_features': 'Max Features per Split',
    'lime_time': 'LIME',
    'shap_values_time': 'TreeSHAP',
    'lmdi_plus_time': 'LMDI+'
})
display_df

,OpenML Data ID,# of Features,LIME,TreeSHAP,LMDI+
13,361260,15,0.082713,0.002666,0.010204
49,361259,32,0.129364,0.003305,0.021501
85,361242,81,0.273451,0.003984,0.052962


In [44]:
# round to fourth decimal place
display_df = display_df.round(4)

In [45]:
# get display_df in markdown format
markdown_df = display_df.to_markdown(index=False)
print(markdown_df)

|   OpenML Data ID |   # of Features |   LIME |   TreeSHAP |   LMDI+ |
|-----------------:|----------------:|-------:|-----------:|--------:|
|           361260 |              15 | 0.0827 |     0.0027 |  0.0102 |
|           361259 |              32 | 0.1294 |     0.0033 |  0.0215 |
|           361242 |              81 | 0.2735 |     0.004  |  0.053  |
